# SpamShield — Train your SMS spam model
Run the cells from top to bottom in Google Colab. The notebook downloads the public UCI SMS Spam Collection, trains TF-IDF + Logistic Regression, evaluates it, and downloads `spamshield_model.joblib`.

In [ ]:
!pip -q install pandas scikit-learn joblib


In [ ]:
import urllib.request, zipfile, os
import pandas as pd

url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
urllib.request.urlretrieve(url, 'sms_dataset.zip')
with zipfile.ZipFile('sms_dataset.zip', 'r') as z:
    z.extractall('sms_data')

df = pd.read_csv('sms_data/SMSSpamCollection', sep='\t', header=None,
                 names=['label', 'message'], encoding='utf-8')
df = df.dropna(subset=['label', 'message'])
df = df[df['label'].isin(['ham', 'spam'])]
print('Total messages:', len(df))
print(df['label'].value_counts())
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

X = df['message'].astype(str)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode',
                              ngram_range=(1, 2), max_features=20000)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced',
                                      random_state=42))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.2%}')
print(classification_report(y_test, predictions))
print('Confusion matrix:\\n', confusion_matrix(y_test, predictions))
joblib.dump(model, 'spamshield_model.joblib')
print('Saved spamshield_model.joblib')


In [ ]:
test_messages = [
    'Congratulations! You have won a free prize. Claim now!',
    'Hey, are we meeting at college tomorrow?'
]
for msg in test_messages:
    label = model.predict([msg])[0]
    probs = model.predict_proba([msg])[0]
    classes = list(model.named_steps['classifier'].classes_)
    spam_score = probs[classes.index('spam')] * 100
    print('Message:', msg)
    print('Prediction:', 'SPAM' if label == 'spam' else 'NOT SPAM')
    print(f'Estimated spam probability: {spam_score:.1f}%\\n')


In [ ]:
from google.colab import files
files.download('spamshield_model.joblib')
